# Prepare well-resolution TARGET2 CellProfiler features

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Not used but needed for downloads from Broad S3
import pyarrow
import fsspec
import s3fs

# ignore mix type warnings from pandas
import warnings

warnings.filterwarnings("ignore")

In [ ]:
def summarize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize NaN, positive infinity, and negative infinity occurrences in a DataFrame.

    :param df: pandas DataFrame to summarize.
    :return: pandas DataFrame summarizing NaN, positive infinity, and negative infinity counts.
    """
    nan_counts = df.isna().sum().sum()
    pos_inf_counts = (df == np.inf).sum().sum()
    neg_inf_counts = (df == -np.inf).sum().sum()

    summary_df = pd.DataFrame({
        'NaN_count': [nan_counts],
        'Pos_inf_count': [pos_inf_counts],
        'Neg_inf_count': [neg_inf_counts]
    })

    return summary_df

def impute_source(source, df):
    df_source = df[df["Metadata_Source"] == source]

    # Separate numeric and non-numeric columns
    numeric_cols = df_source.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df_source.select_dtypes(exclude=[np.number]).columns
    df_numeric = df_source[numeric_cols]
    df_non_numeric = df_source[non_numeric_cols]

    df_numeric = df_numeric.replace([np.inf, -np.inf], np.nan)

    imputer = KNNImputer()

    # doesn't actually update per-column because the imputer does full-df, but that'd be slower so we don't
    with tqdm(total=len(df_numeric), desc=f"Imputing Source: {source}") as pbar:
        imputed_data = imputer.fit_transform(df_numeric)
        pbar.update(len(df_numeric))

    df_imputed_numeric = pd.DataFrame(imputed_data, columns=numeric_cols)

    # Concatenate imputed numeric columns with non-numeric columns
    df_imputed = pd.concat([df_non_numeric.reset_index(drop=True), df_imputed_numeric.reset_index(drop=True)], axis=1)

    return df_imputed

In [ ]:
plates = pd.read_csv("../metadata/plate.csv.gz")
wells = pd.read_csv("../metadata/well.csv.gz")
compounds = pd.read_csv("../metadata/compound.csv.gz")

In [ ]:
profile_formatter = (
    "s3://cellpainting-gallery/cpg0016-jump/"
    "{Metadata_Source}/workspace/profiles/"
    "{Metadata_Batch}/{Metadata_Plate}/{Metadata_Plate}.parquet"
)

loaddata_formatter = (
    "s3://cellpainting-gallery/cpg0016-jump/"
    "{Metadata_Source}/workspace/load_data_csv/"
    "{Metadata_Batch}/{Metadata_Plate}/load_data_with_illum.parquet"
)

In [ ]:
target2_plates = plates.query("Metadata_PlateType == 'TARGET2'")

In [ ]:
target2_plates

## Download well-resolution CellProfiler features for all TARGET2 plates

In [ ]:
%%time

try:
    target2_features = pd.read_parquet("../data/target2_wellres_features.parquet")
except:

    dframes = []
    columns = None # so we get all features

    def fetch_parquet(row):
        s3_path = profile_formatter.format(**row._asdict())
        return pd.read_parquet(s3_path, storage_options={"anon": True}, columns=columns)

    with ThreadPoolExecutor() as executor:
        dframes = list(tqdm(executor.map(fetch_parquet, target2_plates.itertuples(index=False)), total=len(target2_plates)))

    target2_features = pd.concat(dframes)
    target2_features.to_parquet("../data/target2_wellres_features.parquet")

target2_features


## Add batch info again

In [ ]:
target2_features = target2_features.merge(
    target2_plates,
    left_on=["Metadata_Source", "Metadata_Plate"],
    right_on=["Metadata_Source", "Metadata_Plate"],
    how="left"
).drop("Metadata_PlateType", axis=1)

### Impute missing

In [ ]:
summarize_missing_values(target2_features)

In [ ]:
%%time

sources = target2_features["Metadata_Source"].unique()
imputed_dfs = []

# Using ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor(max_workers=len(sources)) as executor:
    # Future to source mapping
    future_to_source = {executor.submit(impute_source, source, target2_features): source for source in sources}

    # Collecting results
    for future in as_completed(future_to_source):
        source = future_to_source[future]
        try:
            imputed_df = future.result()
            imputed_dfs.append(imputed_df)
        except Exception as e:
            print(f"Source {source} generated an exception: {e}")

target2_features_imputed = pd.concat(imputed_dfs, ignore_index=True)

In [ ]:
summarize_missing_values(target2_features_imputed)

In [ ]:
target2_features_imputed

## Add metadata features

In [ ]:
wc = wells.merge(compounds, on="Metadata_JCP2022", how="left")
target2_with_metadata = target2_features_imputed.merge(wc, on=["Metadata_Source", "Metadata_Plate", "Metadata_Well"], how="left")


In [ ]:
target2_with_metadata

## Get standardized compound metadata

In [ ]:
target2_compound_metadata = pd.read_csv(
    "../metadata/repurposing_samples_20200324_standardized.csv.gz",
    compression="gzip"
)
target2_compound_metadata = target2_compound_metadata[["InChIKey_standardized", "SMILES_standardized", "InChI_standardized", "pubchem_cid", "pert_iname"]].drop_duplicates()
target2_compound_metadata.pubchem_cid = pd.to_numeric(target2_compound_metadata.pubchem_cid, errors="coerce").fillna(0).astype(int)

print(target2_compound_metadata.shape)
target2_compound_metadata.head(3)

## Add Broad TARGET2 and CLUE MOA info

In [ ]:
target_moa = pd.read_csv(
    "https://raw.githubusercontent.com/jump-cellpainting/JUMP-MOA/master/JUMP-MOA_compound_metadata.tsv",
    sep="\t"
)
target_moa.pubchem_cid = pd.to_numeric(target_moa.pubchem_cid, errors="coerce").fillna(0).astype(int)

print(target_moa.shape)
target_moa.head(3)

In [ ]:
clue_moa = pd.read_csv(
    "../metadata/repurposing_drugs_20180907.txt",
    sep="\t"
)
clue_moa

## Merge and deal with overlapping columns

In [ ]:
# first create empty pert_iname df, then merge in metadata
df = pd.DataFrame(index=target2_compound_metadata["pert_iname"].str.lower())

target_moa["pert_iname"] = target_moa["pert_iname"].str.lower()
target_moa.set_index("pert_iname", drop=False, inplace=True, verify_integrity=True)

clue_moa["pert_iname"] = clue_moa["pert_iname"].str.lower()
clue_moa.set_index("pert_iname", drop=False, inplace=True, verify_integrity=True)


In [ ]:
df_target = df.merge(target_moa, how="left", right_index=True, left_index=True)
df_target_clue = df_target.merge(clue_moa, how="left", right_index=True, left_index=True)

def merge_moa(row):
    moa_x = row["moa_x"]
    moa_y = row["moa_y"]

    if pd.isna(moa_x) and pd.isna(moa_y):
        return np.nan

    moas = set([moa for moa in [moa_x, moa_y] if not pd.isna(moa)])
    return "|".join(moas) if moas else np.nan

df_target_clue["moa"] = df_target_clue.apply(merge_moa, axis=1)
df_target_clue = df_target_clue[[
    "clinical_phase",
    "target",
    "disease_area",
    "indication",
    "moa",
]]

# Remove all NaN-rows for more efficient downstream 
df_target_clue = df_target_clue.dropna(how="all")

# Add index back as column so we can merge on it
df_target_clue.reset_index(inplace=True)

df_target_clue = df_target_clue.drop_duplicates()

df_target_clue

## Merge into used compounds

In [ ]:
target2_compound_metadata["pert_iname"] = target2_compound_metadata["pert_iname"].str.lower()
target2_compound_metadata = target2_compound_metadata.drop_duplicates()
target2_compound_metadata = target2_compound_metadata.merge(
    df_target_clue,
    how="left",
    left_on="pert_iname",
    right_on="pert_iname"
)

# Have to add 'Metadata' here so that PyCytoMiner doesn't drop the cols later
target2_compound_metadata.columns = [f"Metadata_{col}" if "Metadata_" not in col else col for col in target2_compound_metadata.columns]

target2_compound_metadata

In [ ]:
target2_with_metadata

In [ ]:
target2_complete = target2_with_metadata.merge(
    target2_compound_metadata,
    left_on="Metadata_InChIKey",
    right_on="Metadata_InChIKey_standardized",
    how="left"
).drop_duplicates(subset=["Metadata_Source", "Metadata_Plate", "Metadata_Well"])

target2_complete

## Merge in microscope config

In [ ]:
microscope_config = pd.read_csv("../metadata/microscope_config.csv")
microscope_config["Metadata_Source"] = [f"source_{s}" for s in microscope_config.Metadata_Source.unique()]
microscope_config

In [ ]:
target2_complete = target2_complete.merge(
    microscope_config,
    left_on="Metadata_Source",
    right_on="Metadata_Source",
    how="left"
)

## Save to disk

In [ ]:
target2_complete.to_parquet("../data/target2_wellres_featuresimputed_druginfoadded.parquet")